In [3]:
import torch
from torch_geometric.datasets import QM9

In [4]:
data=QM9(root="data/QM9")

Extracting data/QM9/raw/qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


In [19]:
data[945]

Data(x=[13, 11], edge_index=[2, 26], edge_attr=[26, 4], y=[1, 19], pos=[13, 3], idx=[1], name='gdb_45934', z=[13])

In [ ]:
torch.manual_seed(42)
data=data.shuffle()

In [7]:
tr=data[:100000]
val=data[100000:110000]
te=data[110000:]

In [8]:
from torch_geometric.loader import DataLoader

In [9]:
trd=DataLoader(tr,batch_size=64, shuffle=True)
vald=DataLoader(val,batch_size=64)
ted=DataLoader(te,batch_size=64)

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.nn import global_mean_pool
import torch.nn.init as init

In [38]:
class EdgeConditionedConv(MessagePassing):
    def __init__(self, node_dim, edge_dim, hidden_dim):
        super().__init__(aggr="add")
        self.node_proj = nn.Linear(node_dim, hidden_dim)
        
        # 1. Use a more robust MLP with LeakyReLU
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, hidden_dim * hidden_dim)
        )

        self.update_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # 2. BatchNorm is often better for regression than LayerNorm
        self.norm = nn.BatchNorm1d(hidden_dim)
        self.hidden_dim = hidden_dim

        # 3. INITIALIZATION TRICK: Initialize the last layer of edge_mlp 
        # so it starts near zero (making the transformation stable)
        init.xavier_uniform_(self.edge_mlp[-1].weight, gain=0.01)

    def forward(self, x, edge_index, edge_attr):
        x = self.node_proj(x)
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_j, edge_attr):
        W = self.edge_mlp(edge_attr)
        W = W.view(-1, self.hidden_dim, self.hidden_dim)
        
        # neighbor feature column vector
        x_j = x_j.unsqueeze(-1)
        
        # Edge-conditioned transformation
        message = torch.bmm(W, x_j)
        return message.squeeze(-1)

    def update(self, aggr_out, x):
        # 4. Use a weighted skip connection
        out = x + aggr_out 
        out = self.update_mlp(out)
        return self.norm(out)

In [39]:
class MolecularGNN(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim):
        super().__init__()
        self.conv1 = EdgeConditionedConv(node_dim, edge_dim, hidden_dim)
        self.conv2 = EdgeConditionedConv(hidden_dim, edge_dim, hidden_dim)
        self.conv3 = EdgeConditionedConv(hidden_dim, edge_dim, hidden_dim)

        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.1), # Add dropout to prevent getting stuck
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.conv1(x, edge_index, edge_attr)
        x = self.conv2(x, edge_index, edge_attr)
        x = self.conv3(x, edge_index, edge_attr)
        
        x = global_mean_pool(x, batch)
        return self.readout(x)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MolecularGNN(
    node_dim=data.num_node_features,
    edge_dim=data.num_edge_features,
    hidden_dim=64 # Reduced to 64 for memory efficiency
).to(device)

In [23]:
target_idx = 0 

In [33]:
all_targets = [data.y[:, target_idx] for data in tr]
all_targets = torch.cat(all_targets)
mean = all_targets.mean()
std = all_targets.std()

In [34]:
def train():
    model.train()
    total_loss = 0
    for data in trd:
        data = data.to(device)
        optimizer.zero_grad()
        
        # --- FIX 2: NORMALIZE TARGETS DURING TRAINING ---
        target = (data.y[:, target_idx].unsqueeze(1) - mean) / std
        
        pred = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = loss_fn(pred, target)
        
        loss.backward()
        
        # --- FIX 3: GRADIENT CLIPPING ---
        # Prevents weights from exploding due to the matrix multiplications
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(trd)


In [35]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        # Normalize target for consistent validation loss
        target = (data.y[:, target_idx].unsqueeze(1) - mean) / std
        
        pred = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = loss_fn(pred, target)
        total_loss += loss.item()
    return total_loss / len(loader)

In [40]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# Reduces LR when the loss stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [41]:
for epoch in range(50):
    train_loss = train()
    val_loss = evaluate(vald)
    
    # Step the scheduler
    scheduler.step(val_loss)
    
    curr_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:02d} | LR: {curr_lr:.6f} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

Epoch 00 | LR: 0.001000 | Train: 0.6737 | Val: 0.6103
Epoch 01 | LR: 0.001000 | Train: 0.6359 | Val: 0.5990
Epoch 02 | LR: 0.001000 | Train: 0.6274 | Val: 0.5755
Epoch 03 | LR: 0.001000 | Train: 0.5980 | Val: 0.5173
Epoch 04 | LR: 0.001000 | Train: 0.5350 | Val: 0.4834
Epoch 05 | LR: 0.001000 | Train: 0.5033 | Val: 0.4614
Epoch 06 | LR: 0.001000 | Train: 0.4766 | Val: 0.4404
Epoch 07 | LR: 0.001000 | Train: 0.4558 | Val: 0.4156
Epoch 08 | LR: 0.001000 | Train: 0.4321 | Val: 0.4273
Epoch 09 | LR: 0.001000 | Train: 0.4152 | Val: 0.4019
Epoch 10 | LR: 0.001000 | Train: 0.4000 | Val: 0.3743
Epoch 11 | LR: 0.001000 | Train: 0.3847 | Val: 0.3621
Epoch 12 | LR: 0.001000 | Train: 0.3715 | Val: 0.3680
Epoch 13 | LR: 0.001000 | Train: 0.3640 | Val: 0.3557
Epoch 14 | LR: 0.001000 | Train: 0.3569 | Val: 0.3516
Epoch 15 | LR: 0.001000 | Train: 0.3496 | Val: 0.3537
Epoch 16 | LR: 0.001000 | Train: 0.3426 | Val: 0.3401
Epoch 17 | LR: 0.001000 | Train: 0.3402 | Val: 0.3434
Epoch 18 | LR: 0.001000 | Tr